# STIR-Net V1 — Notebook 29
## Short spatial training + oracle decomposition ladder

This notebook is deliberately **not** another long end-to-end experiment.

It has two jobs:

1. Train the newly updated STIR-Net V1 for a **bounded, moderate amount of time** on the existing first-overfit scene, primarily in spatial-only mode.
2. Decompose the resulting spatial system into progressively less-oracle stages so we can identify the **first interface where performance collapses**.

Designed against repository head:

`ba6bf016e6903f6c0a521ffa2931941b649e6a22`

which contains both the anchor-local native mask update and the event-aware competitive temporal-routing update.

## Causal ladder

```text
GT EVERYTHING
      ↓
Can final local mask decoder work?
      ↓
replace controlled mask hypothesis with actual query
      ↓
Can query representation support masking?
      ↓
replace GT center with predicted proposal anchor
      ↓
Are proposal locations sufficient?
      ↓
replace GT proposal selection with learned existence selection
      ↓
Can model choose one hypothesis per cell?
      ↓
inspect learned proposal score field vs oracle score field
      ↓
Can the spatial network discover cells?
      ↓
screen the same model with temporal information added
```

A neural hidden vector such as D0 or a query embedding has no literal GT value. Therefore oracle boundaries are placed at semantic variables that do have GT: existence, center, assignment, instance mask, and proposal-center field.

For the first mask gate, the local mask decoder is fitted with **GT anchors and one common query vector** while the spatial features are frozen. This tests whether the local spatial evidence + decoder architecture can represent the nine cell masks without relying on per-cell query memorization.

### Runtime policy

- Short training wall budget: **~70 min maximum**
- Whole-notebook hard budget: **~115 min**
- Every major phase is error-isolated.
- Errors are written to `errors.jsonl`.
- Stage checkpoints are saved.
- CUDA OOMs in diagnostic phases are caught and the notebook continues.
- No long overnight training is started.
- Napari is **not opened automatically**.


In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Any, Callable
import copy, gc, json, math, shutil, subprocess, time, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment

from learned.stirnet import StirNet
from learned.stirnet.debugging.acceptance.first_overfit import _reduced_config, _repo_root, build_real_batch
from learned.stirnet.model.matcher import target_ids
from learned.stirnet.model.query_builder import QUERY_SPATIAL_PROPOSAL
from learned.stirnet.training.checkpoint import load_checkpoint, save_checkpoint
from learned.stirnet.training.trainer import Trainer, model_forward_from_batch, move_batch_to_device, move_to_device

EXPECTED_HEAD='ba6bf016e6903f6c0a521ffa2931941b649e6a22'
SEED=40266
SOURCE_ID=9
AMP_DTYPE=torch.float16
TRAIN_WALL_MINUTES=70.0
NOTEBOOK_HARD_MINUTES=115.0
QUERY_BOOTSTRAP_MAX_STEPS=15
QUERY_BOOTSTRAP_MAX_MINUTES=15.0
LOCAL_MASK_BOOTSTRAP_MAX_STEPS=40
LOCAL_MASK_BOOTSTRAP_MAX_MINUTES=35.0
JOINT_MAX_STEPS=20
JOINT_MAX_MINUTES=20.0
ORACLE_MASK_FIT_MAX_STEPS=70
ORACLE_MASK_FIT_MAX_MINUTES=8.0
ORACLE_MASK_FIT_LR=2e-3
SOURCE9_SELECTION_NEIGHBORHOOD_DREF=1.25
DEFAULT_EXIST_THRESHOLD=0.50
MAX_SOURCE9_RENDER_QUERIES=32
OPEN_NAPARI_AT_END=False

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type!='cuda': raise RuntimeError('Notebook 29 requires CUDA.')
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision('high'); torch.backends.cuda.matmul.allow_tf32=True
NOTEBOOK_STARTED=time.monotonic(); NOTEBOOK_DEADLINE=NOTEBOOK_STARTED+NOTEBOOK_HARD_MINUTES*60; TRAIN_DEADLINE=NOTEBOOK_STARTED+TRAIN_WALL_MINUTES*60
REPO_ROOT=_repo_root(Path.cwd())
DATA_DIR=REPO_ROOT/'data'/'learned'/'stirnet'/'first_overfit'/'BlastoSPIM1_F22_030_034'
RUN_DIR=REPO_ROOT/'runs'/'stirnet'/'experiments'/f"29_oracle_ladder_{time.strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True,exist_ok=False)
LOG_PATH=RUN_DIR/'run.log'; RESULTS_JSONL=RUN_DIR/'results.jsonl'; ERRORS_JSONL=RUN_DIR/'errors.jsonl'; TRAIN_JSONL=RUN_DIR/'training.jsonl'
print('Repository :',REPO_ROOT); print('Data       :',DATA_DIR); print('Run dir    :',RUN_DIR); print('GPU        :',torch.cuda.get_device_name(0))


In [ ]:
def _jsonable(value):
    if isinstance(value,Path): return str(value)
    if isinstance(value,np.generic): return value.item()
    if torch.is_tensor(value):
        if value.numel()==1: return value.detach().cpu().item()
        return value.detach().cpu().tolist()
    if isinstance(value,float) and not math.isfinite(value): return None
    return value

def log(message):
    text=f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {message}"; print(text,flush=True)
    with LOG_PATH.open('a',encoding='utf-8') as h: h.write(text+'\n'); h.flush()

def append_jsonl(path,payload):
    with path.open('a',encoding='utf-8') as h:
        json.dump({k:_jsonable(v) for k,v in payload.items()},h); h.write('\n'); h.flush()

def record_result(phase,**payload): append_jsonl(RESULTS_JSONL,{'time':time.strftime('%Y-%m-%d %H:%M:%S'),'phase':phase,**payload})
def record_error(phase,exc):
    text=''.join(traceback.format_exception(type(exc),exc,exc.__traceback__)); log(f"ERROR [{phase}] {type(exc).__name__}: {exc}")
    append_jsonl(ERRORS_JSONL,{'time':time.strftime('%Y-%m-%d %H:%M:%S'),'phase':phase,'type':type(exc).__name__,'message':str(exc),'traceback':text})
def cleanup(): gc.collect(); torch.cuda.empty_cache()
def minutes_left(): return max(0.0,(NOTEBOOK_DEADLINE-time.monotonic())/60.0)
def safe_phase(name,fn,default=None):
    log(f'START {name}')
    try:
        value=fn(); log(f'DONE  {name}'); return value
    except BaseException as exc:
        record_error(name,exc); cleanup(); return default

def git_head():
    try: return subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_ROOT,text=True).strip()
    except Exception: return 'unknown'
HEAD=git_head(); log(f'Git HEAD: {HEAD}')
if HEAD!=EXPECTED_HEAD: log(f'WARNING: local HEAD differs from designed commit {EXPECTED_HEAD}; continuing with local code.')
free_gib=shutil.disk_usage(REPO_ROOT.anchor).free/1024**3; log(f'Free disk: {free_gib:.2f} GiB')
if free_gib<2.0: raise RuntimeError('Less than 2 GiB free disk; refusing unattended run.')
record_result('environment',git_head=HEAD,expected_head=EXPECTED_HEAD,gpu=torch.cuda.get_device_name(0),free_disk_gib=free_gib)


## 1. Build the real all-cell overfit scene and identify semantic cases

In [ ]:
batch_cpu,sample_info=build_real_batch(DATA_DIR); target=batch_cpu['targets'][0]
gt_labels_native=torch.as_tensor(target['label_map']).detach().cpu().numpy().astype(np.int32,copy=False)
current_labels_native=batch_cpu['instance_labels'][0].detach().cpu().numpy().astype(np.int32,copy=False)
spacing_native=batch_cpu['spacing_um'][0].detach().cpu().numpy().astype(np.float64); dref_um=float(batch_cpu['dref_um'][0])
all_gt_ids=target_ids(target).detach().cpu().numpy().astype(int); all_gt_centers=torch.as_tensor(target['centers_cellscale']).detach().cpu().float()
source9_gt_ids=np.unique(gt_labels_native[current_labels_native==SOURCE_ID]); source9_gt_ids=source9_gt_ids[source9_gt_ids>0].astype(int); source9_id_set=set(source9_gt_ids.tolist())
source9_gt_indices=torch.tensor([i for i,g in enumerate(all_gt_ids.tolist()) if int(g) in source9_id_set],dtype=torch.long); source9_gt_centers=all_gt_centers[source9_gt_indices]; source9_gt_ids=all_gt_ids[source9_gt_indices.numpy()].astype(int); source9_id_set=set(source9_gt_ids.tolist())
missing_gt_ids=[]
for gt_id in all_gt_ids.tolist():
    if not np.any(current_labels_native[gt_labels_native==int(gt_id)]>0): missing_gt_ids.append(int(gt_id))
noisy_source_ids=[]
for sid in np.unique(current_labels_native):
    if int(sid)>0 and not np.any(gt_labels_native[current_labels_native==int(sid)]>0): noisy_source_ids.append(int(sid))
log(f"Scene current={sample_info['current_count']} GT={sample_info['target_count']} shape={sample_info['roi_shape']} dref={dref_um:.3f}um")
log(f'Source-9 GT IDs: {source9_gt_ids.tolist()}'); log(f'Missing GT IDs: {missing_gt_ids}'); log(f'Noisy source IDs: {noisy_source_ids}')
record_result('scene',sample_info=sample_info,source9_gt_ids=source9_gt_ids.tolist(),missing_gt_ids=missing_gt_ids,noisy_source_ids=noisy_source_ids)


## 2. Build a strict spatial-only batch

In [ ]:
NODE_KEYS={'graph_x','tracklet_id','node_instance_grid','node_history_valid','node_observed_ref_um','node_time_offset','node_ids','node_event_features'}
EDGE_INDEX_KEYS={'graph_edge_index','accepted_association_edge_index'}; EDGE_ATTR_KEYS={'graph_edge_attr','accepted_association_edge_attr'}
TRACKLET_KEYS={'temporal_ref_um','temporal_status','temporal_batch','history_support','history_support_valid','history_support_dt','history_support_center_um','history_support_extent_um','best_current_component_id','best_component_overlap','second_best_component_overlap'}
HYP_EDGE_INDEX_KEYS={'hypothesis_edge_index'}; HYP_EDGE_ATTR_KEYS={'hypothesis_edge_attr'}
def make_spatial_only_batch(batch):
    out=dict(batch)
    for key in NODE_KEYS|EDGE_ATTR_KEYS|TRACKLET_KEYS|HYP_EDGE_ATTR_KEYS:
        value=out.get(key)
        if torch.is_tensor(value): out[key]=value[:0]
    for key in EDGE_INDEX_KEYS|HYP_EDGE_INDEX_KEYS:
        value=out.get(key)
        if torch.is_tensor(value): out[key]=value[:,:0]
    return out
spatial_batch_cpu=make_spatial_only_batch(batch_cpu)
assert spatial_batch_cpu['graph_x'].shape[0]==0 and spatial_batch_cpu['temporal_ref_um'].shape[0]==0
b_spatial=move_batch_to_device(spatial_batch_cpu,device); b_spatial['spatial_inputs']=b_spatial['spatial_inputs'].to(dtype=AMP_DTYPE); b_spatial['instance_labels']=b_spatial['instance_labels'].to(dtype=torch.int32)
log('Spatial-only batch constructed with zero temporal nodes/tracklets.')


## 3. Warm-start the updated V1 model and configure the bounded curriculum

In [ ]:
def discover_warm_start():
    candidates=list(REPO_ROOT.glob('runs/stirnet/overnight/27_overnight_*/checkpoint_best_spatial_query.pt'))
    candidates+=list(REPO_ROOT.glob('runs/stirnet/**/checkpoint_best_spatial_query.pt'))
    candidates=[p for p in set(candidates) if p.exists() and '29_oracle_ladder_' not in str(p)]
    return max(candidates,key=lambda p:p.stat().st_mtime) if candidates else None

cfg=_reduced_config(); cfg.proposals.enabled=True; cfg.proposals.query_mode='spatial_proposals'; cfg.local_masks.enabled=True
cfg.curriculum.enabled=True; cfg.curriculum.spatial_dense_steps=0; cfg.curriculum.temporal_dense_steps=0
cfg.curriculum.query_bootstrap_steps=QUERY_BOOTSTRAP_MAX_STEPS; cfg.curriculum.native_bootstrap_steps=LOCAL_MASK_BOOTSTRAP_MAX_STEPS
cfg.curriculum.joint_spatial_lr_scale=0.10; cfg.curriculum.joint_dense_lr_scale=0.50
model=StirNet(cfg).to(device); WARM_START_CHECKPOINT=discover_warm_start(); warm_start_info={'checkpoint':None,'loaded':False,'migration':[]}
if WARM_START_CHECKPOINT is not None:
    log(f'Warm start candidate: {WARM_START_CHECKPOINT}')
    try:
        loaded=load_checkpoint(WARM_START_CHECKPOINT,model,optimizer=None,scheduler=None,scaler=None,map_location='cpu',strict=True,migrate_history=True)
        warm_start_info={'checkpoint':str(WARM_START_CHECKPOINT),'loaded':True,'step':loaded.get('step'),'epoch':loaded.get('epoch'),'migration':loaded.get('model_migration',[])}
        log(f"Warm start loaded step={loaded.get('step')} with {len(warm_start_info['migration'])} migration notes")
    except BaseException as exc:
        record_error('warm_start',exc); log('Continuing from fresh initialization.')
else: log('No previous spatial-query checkpoint found; using fresh initialization.')
trainer=Trainer(model,cfg,device=device,amp_dtype='fp16'); trainer.global_step=0
save_checkpoint(RUN_DIR/'checkpoint_initial.pt',model=trainer.model,step=0,config=cfg,extra={'warm_start':warm_start_info})
record_result('warm_start',**warm_start_info)


## 4. Preflight

In [ ]:
def _run_preflight():
    trainer.model.eval(); torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(),torch.autocast(device_type='cuda',dtype=AMP_DTYPE):
        outputs=model_forward_from_batch(trainer.model,b_spatial,return_debug=False)
        losses=trainer.criterion(outputs,b_spatial['targets'],local_mask_decoder=trainer.model.local_mask_decoder)
    result={k:float(v.detach().cpu()) for k,v in losses.items()}; result['peak_gib']=torch.cuda.max_memory_allocated()/1024**3
    log(f"Preflight loss={result.get('loss',float('nan')):.4f} dice_hi={result.get('dice_hi',float('nan')):.4f} coarse={result.get('dice_coarse',float('nan')):.4f} peak={result['peak_gib']:.2f}GiB")
    return result
preflight=safe_phase('preflight',_run_preflight)
if preflight is not None: record_result('preflight',**preflight)
cleanup()


# Part A — Bounded short training

We intentionally do **not** retrain every stage from scratch. The warm-start checkpoint already contains useful spatial/proposal/query knowledge. The updated model gets enough optimization to exercise the changed semantics:

1. query refresh — occupancy-preserving coarse targets + immutable-anchor center semantics
2. local mask bootstrap — only `local_mask_decoder` trainable
3. short joint spatial/local adaptation — local-mask gradients can reach D0 with low spatial LR

Training is spatial-only. Temporal routing is screened later.


In [ ]:
training_rows=[]
def persist_training_row(row): training_rows.append(row); append_jsonl(TRAIN_JSONL,row)
def stage_checkpoint(name):
    save_checkpoint(RUN_DIR/f'checkpoint_{name}.pt',model=trainer.model,step=trainer.global_step,config=cfg,extra={'stage':name,'warm_start':warm_start_info})
    save_checkpoint(RUN_DIR/'checkpoint_last.pt',model=trainer.model,step=trainer.global_step,config=cfg,extra={'stage':name})
def run_training_stage(name,stage_start_step,stage_end_step,max_steps,max_minutes):
    if time.monotonic()>=TRAIN_DEADLINE: return {'stage':name,'trained_steps':0,'reason':'global_train_deadline'}
    trainer.global_step=int(stage_start_step); started=time.monotonic(); deadline=min(started+max_minutes*60,TRAIN_DEADLINE); completed=0; last_metrics=None
    log(f'TRAIN {name}: start={trainer.global_step} max_steps={max_steps} max_minutes={max_minutes}')
    while completed<max_steps and time.monotonic()<deadline and (stage_end_step is None or trainer.global_step<stage_end_step):
        t0=time.monotonic()
        try: metrics=trainer.train_step(b_spatial)
        except BaseException as exc:
            record_error(f'train_{name}',exc); cleanup(); break
        completed+=1; last_metrics=metrics; row={'time':time.strftime('%Y-%m-%d %H:%M:%S'),'stage':trainer.curriculum_stage.name,'requested_stage':name,'step':trainer.global_step,'step_seconds':time.monotonic()-t0,**metrics}; persist_training_row(row)
        if completed==1 or completed%5==0: log(f"{name} n={completed:03d} global={trainer.global_step:04d} loss={metrics.get('loss',float('nan')):.4f} dice_hi={metrics.get('dice_hi',float('nan')):.4f} coarse={metrics.get('dice_coarse',float('nan')):.4f} center={metrics.get('center',float('nan')):.4f}")
        if completed%10==0: stage_checkpoint(f'{name}_progress')
    stage_checkpoint(name)
    result={'stage':name,'trained_steps':completed,'elapsed_minutes':(time.monotonic()-started)/60,'final_global_step':trainer.global_step,'last_metrics':last_metrics,'reason':'stage_complete' if completed>=max_steps else 'time_or_error_stop'}; record_result('training_stage',**result); return result
stage_summaries=[]
stage_summaries.append(run_training_stage('query_refresh',0,QUERY_BOOTSTRAP_MAX_STEPS,QUERY_BOOTSTRAP_MAX_STEPS,QUERY_BOOTSTRAP_MAX_MINUTES))
native_start=QUERY_BOOTSTRAP_MAX_STEPS; native_end=native_start+LOCAL_MASK_BOOTSTRAP_MAX_STEPS
stage_summaries.append(run_training_stage('local_mask_bootstrap',native_start,native_end,LOCAL_MASK_BOOTSTRAP_MAX_STEPS,LOCAL_MASK_BOOTSTRAP_MAX_MINUTES))
stage_summaries.append(run_training_stage('short_joint',native_end,None,JOINT_MAX_STEPS,JOINT_MAX_MINUTES))
save_checkpoint(RUN_DIR/'checkpoint_short_train_final.pt',model=trainer.model,step=trainer.global_step,config=cfg,extra={'stage_summaries':stage_summaries,'warm_start':warm_start_info})
if training_rows: pd.DataFrame(training_rows).to_csv(RUN_DIR/'training.csv',index=False)
display(pd.DataFrame(stage_summaries)); log(f'Short training finished after {(time.monotonic()-NOTEBOOK_STARTED)/60:.1f} notebook minutes.')


# Part B — Spatial-only model snapshot

In [ ]:
def _forward_spatial_snapshot():
    trainer.model.eval(); cleanup(); torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(),torch.autocast(device_type='cuda',dtype=AMP_DTYPE): outputs=model_forward_from_batch(trainer.model,b_spatial,return_debug=True)
    log(f"Spatial snapshot Q={outputs.exist_logits.shape[1]} coarse={tuple(outputs.coarse_mask_logits.shape[-3:])} peak={torch.cuda.max_memory_allocated()/1024**3:.2f}GiB"); return outputs
out_spatial=safe_phase('spatial_snapshot',_forward_spatial_snapshot)
if out_spatial is None: raise RuntimeError('Critical spatial snapshot failed. See errors.jsonl.')


In [ ]:
def valid_proposal_query_rows(outputs):
    valid=~outputs.query_padding_mask[0]; proposal=outputs.query_types[0]==QUERY_SPATIAL_PROPOSAL
    return torch.nonzero(valid&proposal,as_tuple=False).flatten()
def build_gt_aware_pairing(outputs,gt_ids_ordered,gt_centers_ordered):
    qrows=valid_proposal_query_rows(outputs); refs=outputs.query_initial_references_cellscale[0,qrows].detach().float().cpu()
    if len(qrows)==0 or len(gt_centers_ordered)==0: return {'query_rows':torch.empty(0,dtype=torch.long),'gt_rows':torch.empty(0,dtype=torch.long),'gt_ids':np.asarray([],dtype=int),'distances_dref':torch.empty(0)}
    dist=torch.cdist(refs,gt_centers_ordered.detach().float().cpu()); q_np,g_np=linear_sum_assignment(dist.numpy()); qlocal=torch.as_tensor(q_np); grows=torch.as_tensor(g_np); order=torch.argsort(grows); qlocal=qlocal[order]; grows=grows[order]
    return {'query_rows':qrows.detach().cpu()[qlocal],'gt_rows':grows,'gt_ids':np.asarray(gt_ids_ordered,dtype=int)[grows.numpy()],'distances_dref':dist[qlocal,grows]}
source9_pairing=build_gt_aware_pairing(out_spatial,source9_gt_ids,source9_gt_centers)
pairing_df=pd.DataFrame({'gt_id':source9_pairing['gt_ids'],'query_row':source9_pairing['query_rows'].numpy(),'initial_anchor_error_dref':source9_pairing['distances_dref'].numpy()})
if len(pairing_df):
    qdev=source9_pairing['query_rows'].to(device); grows=source9_pairing['gt_rows']; paired_gt=source9_gt_centers[grows]
    pairing_df['source_instance_id']=out_spatial.source_instance_ids[0,qdev].detach().cpu().numpy(); pairing_df['exist_prob']=out_spatial.exist_logits[0,qdev].sigmoid().detach().cpu().numpy(); final=out_spatial.centers_cellscale[0,qdev].detach().float().cpu(); pairing_df['final_center_error_dref']=torch.linalg.vector_norm(final-paired_gt,dim=-1).numpy()
display(pairing_df); pairing_df.to_csv(RUN_DIR/'00_source9_gt_aware_pairing.csv',index=False)


In [ ]:
def eval_local_masks(decoder,outputs,gt_ids_ordered,anchors_cellscale,query_embeddings,tag):
    rows=[]
    for i,(gt_id,anchor) in enumerate(zip(gt_ids_ordered,anchors_cellscale)):
        with torch.no_grad(),torch.autocast(device_type='cuda',dtype=AMP_DTYPE):
            pred=decoder.decode_one(outputs.d0_features,outputs.spatial_inputs,outputs.dense_outputs,query_embeddings[i],anchor.to(device),outputs.spacing_um[0],outputs.dref_um[0],batch_index=0)
        if pred.logits is None: raise RuntimeError('Local decoder returned no logits')
        target_crop=torch.as_tensor(gt_labels_native[pred.slices]==int(gt_id),dtype=torch.bool,device=device); support=pred.support; prob=pred.logits.float().sigmoid(); hard=prob>=0.5
        p=prob*support.float(); h=hard&support; target_total=float(np.count_nonzero(gt_labels_native==int(gt_id))); inside=float((target_crop&support).sum().cpu())
        inter_soft=float((p*target_crop.float()).sum().cpu()); pred_soft=float(p.sum().cpu()); soft=(2*inter_soft+1e-6)/(pred_soft+target_total+1e-6)
        inter_hard=float((h&target_crop).sum().cpu()); pred_hard=float(h.sum().cpu()); hard_dice=(2*inter_hard+1e-6)/(pred_hard+target_total+1e-6)
        rows.append({'tag':tag,'gt_id':int(gt_id),'support_coverage':inside/max(target_total,1),'soft_dice_full':soft,'hard_dice_full':hard_dice,'hard_volume_ratio':pred_hard/max(target_total,1),'target_voxels':int(target_total),'predicted_voxels':int(pred_hard)})
    frame=pd.DataFrame(rows); summary={'tag':tag,'cell_count':len(frame),'soft_dice_mean':float(frame.soft_dice_full.mean()) if len(frame) else float('nan'),'soft_dice_min':float(frame.soft_dice_full.min()) if len(frame) else float('nan'),'hard_dice_mean':float(frame.hard_dice_full.mean()) if len(frame) else float('nan'),'hard_dice_min':float(frame.hard_dice_full.min()) if len(frame) else float('nan'),'support_coverage_min':float(frame.support_coverage.min()) if len(frame) else float('nan'),'volume_ratio_mean':float(frame.hard_volume_ratio.mean()) if len(frame) else float('nan')}; return frame,summary

def local_training_loss(decoder,outputs,gt_id,anchor,query_embedding):
    with torch.autocast(device_type='cuda',dtype=AMP_DTYPE): pred=decoder.decode_one(outputs.d0_features.detach(),outputs.spatial_inputs.detach(),{k:v.detach() for k,v in outputs.dense_outputs.items()},query_embedding,anchor,outputs.spacing_um[0],outputs.dref_um[0],batch_index=0)
    target=torch.as_tensor(gt_labels_native[pred.slices]==int(gt_id),dtype=torch.float32,device=device); support=pred.support; logits=pred.logits.float()[support]; tgt=target[support]; prob=logits.sigmoid(); inter=(prob*tgt).sum(); dice=(2*inter+1e-6)/(prob.sum()+tgt.sum()+1e-6); bce=F.binary_cross_entropy_with_logits(logits,tgt); return (1-dice)+0.5*bce


# Gate 1 — GT EVERYTHING: can the local mask mechanism work?

In [ ]:
gate1_frame=pd.DataFrame(); gate1_summary=None
def _gate1():
    decoder=copy.deepcopy(trainer.model.local_mask_decoder).to(device); decoder.train(); opt=torch.optim.AdamW(decoder.parameters(),lr=ORACLE_MASK_FIT_LR,weight_decay=1e-4); common=torch.zeros(cfg.decoder.d_model,device=device); anchors=source9_gt_centers.to(device); ids=source9_gt_ids.tolist(); deadline=min(time.monotonic()+ORACLE_MASK_FIT_MAX_MINUTES*60,NOTEBOOK_DEADLINE); best=-1.; best_state=None; hist=[]
    for step in range(ORACLE_MASK_FIT_MAX_STEPS+1):
        if time.monotonic()>=deadline: break
        decoder.train(); opt.zero_grad(set_to_none=True); loss=torch.stack([local_training_loss(decoder,out_spatial,int(g),a,common) for g,a in zip(ids,anchors)]).mean()
        if step<ORACLE_MASK_FIT_MAX_STEPS: loss.backward(); torch.nn.utils.clip_grad_norm_(decoder.parameters(),1.0); opt.step()
        if step==0 or step%10==0:
            decoder.eval(); frame,summary=eval_local_masks(decoder,out_spatial,ids,anchors,common[None].expand(len(ids),-1),'G1_oracle_common_query'); hist.append({'step':step,'loss':float(loss.detach().cpu()),**summary}); log(f"G1 step={step:03d} loss={float(loss.detach().cpu()):.4f} hardDice={summary['hard_dice_mean']:.4f} min={summary['hard_dice_min']:.4f}")
            if summary['hard_dice_mean']>best: best=summary['hard_dice_mean']; best_state=copy.deepcopy(decoder.state_dict())
    if best_state is not None: decoder.load_state_dict(best_state)
    decoder.eval(); frame,summary=eval_local_masks(decoder,out_spatial,ids,anchors,common[None].expand(len(ids),-1),'G1_oracle_common_query'); pd.DataFrame(hist).to_csv(RUN_DIR/'G1_oracle_mask_fit_history.csv',index=False); frame.to_csv(RUN_DIR/'G1_oracle_mask_per_cell.csv',index=False); return frame,summary
gate1_result=safe_phase('G1_oracle_mask_fit',_gate1)
if gate1_result is not None: gate1_frame,gate1_summary=gate1_result; display(gate1_frame); print(gate1_summary); record_result('G1_oracle_mask_fit',**gate1_summary)
cleanup()


# Gate 2 — Replace controlled hypothesis with actual learned query

In [ ]:
gate2_frame=pd.DataFrame(); gate2_summary=None
def _gate2():
    q=source9_pairing['query_rows']; g=source9_pairing['gt_rows']; qd=q.to(device); ids=source9_pairing['gt_ids']; anchors=source9_gt_centers[g].to(device); emb=out_spatial.query_embeddings[0,qd].detach(); frame,summary=eval_local_masks(trainer.model.local_mask_decoder,out_spatial,ids,anchors,emb,'G2_actual_query_gt_anchor'); frame.to_csv(RUN_DIR/'G2_actual_query_gt_anchor.csv',index=False); return frame,summary
gate2_result=safe_phase('G2_actual_query_gt_anchor',_gate2)
if gate2_result is not None: gate2_frame,gate2_summary=gate2_result; display(gate2_frame); print(gate2_summary); record_result('G2_actual_query_gt_anchor',**gate2_summary)


# Gate 3 — Replace GT centers with actual immutable proposal anchors

In [ ]:
gate3_frame=pd.DataFrame(); gate3_summary=None; center_identity_df=pd.DataFrame()
def _gate3():
    q=source9_pairing['query_rows']; g=source9_pairing['gt_rows']; qd=q.to(device); ids=source9_pairing['gt_ids']; anchors=out_spatial.query_initial_references_cellscale[0,qd].detach(); emb=out_spatial.query_embeddings[0,qd].detach(); frame,summary=eval_local_masks(trainer.model.local_mask_decoder,out_spatial,ids,anchors,emb,'G3_actual_query_predicted_anchor')
    gt=source9_gt_centers[g]; initial=anchors.float().cpu(); final=out_spatial.centers_cellscale[0,qd].detach().float().cpu(); ie=torch.linalg.vector_norm(initial-gt,dim=-1); fe=torch.linalg.vector_norm(final-gt,dim=-1); mv=torch.linalg.vector_norm(final-initial,dim=-1)
    cdf=pd.DataFrame({'gt_id':ids,'query_row':q.numpy(),'initial_error_dref':ie.numpy(),'final_error_dref':fe.numpy(),'movement_dref':mv.numpy()}); summary.update({'initial_center_error_mean_dref':float(ie.mean()),'final_center_error_mean_dref':float(fe.mean()),'center_movement_mean_dref':float(mv.mean())}); frame.to_csv(RUN_DIR/'G3_predicted_anchor_masks.csv',index=False); cdf.to_csv(RUN_DIR/'G3_center_identity.csv',index=False); return frame,summary,cdf
gate3_result=safe_phase('G3_predicted_anchor',_gate3)
if gate3_result is not None: gate3_frame,gate3_summary,center_identity_df=gate3_result; display(gate3_frame); display(center_identity_df); print(gate3_summary); record_result('G3_predicted_anchor',**gate3_summary)


# Gate 4 — Proposal locations only

In [ ]:
def proposal_set_metrics(refs,centers,prefix):
    refs=refs.detach().float().cpu(); centers=centers.detach().float().cpu()
    if len(centers)==0: return {f'{prefix}_proposal_count':len(refs),f'{prefix}_gt_count':0}
    if len(refs)==0: return {f'{prefix}_proposal_count':0,f'{prefix}_gt_count':len(centers),f'{prefix}_recall_0p25':0.,f'{prefix}_recall_0p5':0.,f'{prefix}_recall_1p0':0.,f'{prefix}_nearest_mean_dref':float('inf'),f'{prefix}_nearest_max_dref':float('inf'),f'{prefix}_duplicates_0p5':0}
    d=torch.cdist(refs,centers); n=d.min(dim=0).values; c=(d<=0.5).sum(dim=0)
    return {f'{prefix}_proposal_count':len(refs),f'{prefix}_gt_count':len(centers),f'{prefix}_recall_0p25':float((n<=.25).float().mean()),f'{prefix}_recall_0p5':float((n<=.5).float().mean()),f'{prefix}_recall_1p0':float((n<=1.).float().mean()),f'{prefix}_nearest_mean_dref':float(n.mean()),f'{prefix}_nearest_max_dref':float(n.max()),f'{prefix}_duplicates_0p5':int(torch.clamp(c-1,min=0).sum())}
gate4_summary=None; missing_gt_proposal_df=pd.DataFrame()
def _gate4():
    p=out_spatial.proposals
    if p is None: raise RuntimeError('Missing proposal state')
    valid=~p.padding_mask[0]; learned=valid&~p.fallback_mask[0]; allrefs=p.references_cellscale[0,valid].detach().float().cpu(); lrefs=p.references_cellscale[0,learned].detach().float().cpu(); s={}
    for refs,centers,prefix in [(allrefs,all_gt_centers,'all_gt_all_proposals'),(lrefs,all_gt_centers,'all_gt_learned'),(allrefs,source9_gt_centers,'source9_all_proposals'),(lrefs,source9_gt_centers,'source9_learned')]: s.update(proposal_set_metrics(refs,centers,prefix))
    idrow={int(g):i for i,g in enumerate(all_gt_ids.tolist())}; rows=[]
    for gid in missing_gt_ids:
        c=all_gt_centers[idrow[gid]:idrow[gid]+1]; rows.append({'gt_id':gid,'nearest_learned_proposal_dref':float(torch.cdist(lrefs,c).min()) if len(lrefs) else float('inf'),'nearest_any_proposal_dref':float(torch.cdist(allrefs,c).min()) if len(allrefs) else float('inf')})
    f=pd.DataFrame(rows); f.to_csv(RUN_DIR/'G4_missing_gt_proposals.csv',index=False); pd.DataFrame([s]).to_csv(RUN_DIR/'G4_proposal_location_summary.csv',index=False); return s,f
gate4_result=safe_phase('G4_proposal_locations',_gate4)
if gate4_result is not None: gate4_summary,missing_gt_proposal_df=gate4_result; display(pd.DataFrame([gate4_summary])); display(missing_gt_proposal_df) if len(missing_gt_proposal_df) else None; record_result('G4_proposal_locations',**gate4_summary)


# Gate 5 — Learned proposal selection / cardinality

In [ ]:
def binary_auc_small(scores,labels):
    scores=np.asarray(scores,float); labels=np.asarray(labels,bool); p=scores[labels]; n=scores[~labels]
    if len(p)==0 or len(n)==0:return float('nan')
    return float(((p[:,None]>n[None,:]).astype(float)+.5*(p[:,None]==n[None,:]).astype(float)).mean())
def selection_metrics_for_source9(outputs,threshold):
    q=valid_proposal_query_rows(outputs); refs=outputs.query_initial_references_cellscale[0,q].detach().float().cpu(); scores=outputs.exist_logits[0,q].sigmoid().detach().float().cpu()
    if len(refs)==0:return {'threshold':threshold,'cluster_candidates':0,'selected':0,'unique_gt_covered':0,'missing_gt':len(source9_gt_ids),'duplicate_selected':0,'exactly_one_gt':0}
    d=torch.cdist(refs,source9_gt_centers.float()); nd,ng=d.min(dim=1); cluster=nd<=SOURCE9_SELECTION_NEIGHBORHOOD_DREF; selected=cluster&(scores>=float(threshold)); sd=nd[selected]; sg=ng[selected]; assigned=sg[sd<=1.0]; counts=torch.bincount(assigned,minlength=len(source9_gt_ids)); unique=int((counts>0).sum())
    return {'threshold':float(threshold),'cluster_candidates':int(cluster.sum()),'selected':int(selected.sum()),'selected_assigned_within_1dref':int((sd<=1.).sum()),'unique_gt_covered':unique,'missing_gt':len(source9_gt_ids)-unique,'duplicate_selected':int(torch.clamp(counts-1,min=0).sum()),'exactly_one_gt':int((counts==1).sum()),'mean_selected_exist':float(scores[selected].mean()) if bool(selected.any()) else float('nan')}
gate5_sweep_df=pd.DataFrame(); gate5_summary=None
def _gate5():
    frame=pd.DataFrame([selection_metrics_for_source9(out_spatial,t) for t in (.1,.2,.3,.5,.7,.9)]); q=valid_proposal_query_rows(out_spatial); scores=out_spatial.exist_logits[0,q].sigmoid().detach().cpu().numpy(); positives=set(source9_pairing['query_rows'].tolist()); labels=np.asarray([int(r) in positives for r in q.detach().cpu().tolist()],bool); auc=binary_auc_small(scores,labels); default=frame[np.isclose(frame.threshold,DEFAULT_EXIST_THRESHOLD)].iloc[0].to_dict(); summary={**default,'oracle_pair_ranking_auc_all_proposals':auc,'total_valid_proposal_queries':len(q)}; frame.to_csv(RUN_DIR/'G5_selection_threshold_sweep.csv',index=False); pd.DataFrame([summary]).to_csv(RUN_DIR/'G5_selection_default.csv',index=False); return frame,summary
gate5_result=safe_phase('G5_learned_selection',_gate5)
if gate5_result is not None: gate5_sweep_df,gate5_summary=gate5_result; display(gate5_sweep_df); print(gate5_summary); record_result('G5_learned_selection',**gate5_summary)


# Gate 6 — Proposal score field vs proposal extraction

In [ ]:
def centers_to_native_voxels(centers_cellscale,shape,spacing_um,dref):
    centers=centers_cellscale.to(device=spacing_um.device,dtype=torch.float32); st=torch.tensor(shape,device=spacing_um.device,dtype=torch.float32); extent=(st-1)*spacing_um.float(); cu=centers*dref.float(); vox=torch.round((cu+.5*extent[None])/spacing_um.float()[None].clamp_min(1e-8)).long(); return torch.minimum(torch.maximum(vox,torch.zeros_like(vox)),st.long()[None]-1)
gate6_summary=None; proposal_score_gt_df=pd.DataFrame()
def _gate6():
    score=out_spatial.dense_outputs['proposal_score_logits']; shape=tuple(int(v) for v in score.shape[-3:]); spacing=out_spatial.spacing_um[0]; dref=out_spatial.dref_um[0]; gtdev=all_gt_centers.to(device); vox=centers_to_native_voxels(gtdev,shape,spacing,dref); oracle=torch.full(shape,-20.,device=device); oracle[vox[:,0],vox[:,1],vox[:,2]]=20.; padding=None if b_spatial.get('spatial_padding_mask') is None else b_spatial['spatial_padding_mask'][0]; trainer.model.eval(); ov=trainer.model.spatial_proposal_generator._learned_centers(oracle,spacing,dref,padding,int(cfg.proposals.max_proposals)); our=trainer.model.spatial_proposal_generator._relative_voxel_centers_um(ov,shape,spacing); orefs=(our/dref.float().clamp_min(1e-8)).detach().cpu(); p=out_spatial.proposals; valid=~p.padding_mask[0]; learned=valid&~p.fallback_mask[0]; lrefs=p.references_cellscale[0,learned].detach().float().cpu(); s={}
    for refs,centers,prefix in [(orefs,all_gt_centers,'oracle_score_nms_all_gt'),(orefs,source9_gt_centers,'oracle_score_nms_source9'),(lrefs,all_gt_centers,'learned_score_peaks_all_gt'),(lrefs,source9_gt_centers,'learned_score_peaks_source9')]: s.update(proposal_set_metrics(refs,centers,prefix))
    prob=score[0,0].sigmoid().detach().float().cpu().numpy(); center_scores=prob[vox[:,0].cpu().numpy(),vox[:,1].cpu().numpy(),vox[:,2].cpu().numpy()]; rows=[{'gt_id':int(g),'score_at_gt_center':float(sc),'missing_from_current_mask':int(g) in set(missing_gt_ids),'source9':int(g) in source9_id_set} for g,sc in zip(all_gt_ids.tolist(),center_scores.tolist())]; frame=pd.DataFrame(rows); bg=prob[gt_labels_native==0]
    if bg.size: s.update({'background_score_q50':float(np.quantile(bg,.5)),'background_score_q90':float(np.quantile(bg,.9)),'background_score_q99':float(np.quantile(bg,.99)),'gt_center_score_mean':float(np.mean(center_scores)),'gt_center_score_min':float(np.min(center_scores))})
    frame.to_csv(RUN_DIR/'G6_proposal_scores_at_gt.csv',index=False); pd.DataFrame([s]).to_csv(RUN_DIR/'G6_score_field_summary.csv',index=False); return s,frame
gate6_result=safe_phase('G6_proposal_score_field',_gate6)
if gate6_result is not None: gate6_summary,proposal_score_gt_df=gate6_result; display(pd.DataFrame([gate6_summary])); display(proposal_score_gt_df); record_result('G6_proposal_score_field',**gate6_summary)
cleanup()


# Gate 7 — Missing-cell discovery and noisy-current-component suppression

In [ ]:
missing_discovery_df=pd.DataFrame(); noisy_suppression_df=pd.DataFrame(); gate7_summary=None
def _gate7():
    q=valid_proposal_query_rows(out_spatial); refs=out_spatial.query_initial_references_cellscale[0,q].detach().float().cpu(); scores=out_spatial.exist_logits[0,q].sigmoid().detach().float().cpu(); sids=out_spatial.source_instance_ids[0,q].detach().cpu().long(); idrow={int(g):i for i,g in enumerate(all_gt_ids.tolist())}; mr=[]
    for gid in missing_gt_ids:
        c=all_gt_centers[idrow[gid]:idrow[gid]+1]
        if len(refs):
            d=torch.cdist(refs,c)[:,0]; nr=int(d.argmin()); nearby=d<=1.; sel=nearby&(scores>=DEFAULT_EXIST_THRESHOLD); mr.append({'gt_id':gid,'nearest_proposal_dref':float(d[nr]),'nearest_exist_prob':float(scores[nr]),'proposals_within_1dref':int(nearby.sum()),'selected_within_1dref':int(sel.sum())})
        else: mr.append({'gt_id':gid,'nearest_proposal_dref':float('inf'),'nearest_exist_prob':float('nan'),'proposals_within_1dref':0,'selected_within_1dref':0})
    nr=[]
    for sid in noisy_source_ids:
        tied=sids==sid; sel=tied&(scores>=DEFAULT_EXIST_THRESHOLD); nr.append({'source_id':sid,'proposal_queries':int(tied.sum()),'selected_queries':int(sel.sum()),'max_exist_prob':float(scores[tied].max()) if bool(tied.any()) else float('nan'),'mean_exist_prob':float(scores[tied].mean()) if bool(tied.any()) else float('nan')})
    mf=pd.DataFrame(mr); nf=pd.DataFrame(nr); summary={'missing_gt_count':len(missing_gt_ids),'missing_gt_with_selected_proposal_within_1dref':int((mf.selected_within_1dref>0).sum()) if len(mf) else 0,'noisy_source_count':len(noisy_source_ids),'noisy_sources_with_selected_query':int((nf.selected_queries>0).sum()) if len(nf) else 0}; mf.to_csv(RUN_DIR/'G7_missing_cell_discovery.csv',index=False); nf.to_csv(RUN_DIR/'G7_noisy_component_suppression.csv',index=False); return mf,nf,summary
gate7_result=safe_phase('G7_missing_and_noise',_gate7)
if gate7_result is not None:
    missing_discovery_df,noisy_suppression_df,gate7_summary=gate7_result
    if len(missing_discovery_df): display(missing_discovery_df)
    if len(noisy_suppression_df): display(noisy_suppression_df)
    print(gate7_summary); record_result('G7_missing_and_noise',**gate7_summary)


# Auxiliary source-9 composition test: oracle one-per-GT selection vs learned existence selection

In [ ]:
composition_oracle_summary=None; composition_learned_summary=None
def source9_crop_slices(margin_dref=1.5):
    coords=np.argwhere(current_labels_native==SOURCE_ID)
    if len(coords)==0: coords=np.argwhere(np.isin(gt_labels_native,source9_gt_ids))
    lo=coords.min(axis=0); hi=coords.max(axis=0)+1; margin=np.ceil(margin_dref*dref_um/spacing_native).astype(int); lo=np.maximum(0,lo-margin); hi=np.minimum(np.asarray(gt_labels_native.shape),hi+margin); return tuple(slice(int(a),int(b)) for a,b in zip(lo,hi))
def compose_queries(outputs,qrows):
    qrows=qrows.to(device=device,dtype=torch.long)
    if len(qrows)==0:return None
    rendered=trainer.model.render_masks(outputs,[qrows])[0]; exist=outputs.exist_logits[0,qrows].sigmoid(); prob=rendered.float().sigmoid(); score=prob*exist[:,None,None,None].float(); valid=prob>=.5; score=score.masked_fill(~valid,-torch.inf); best,winner=score.max(dim=0); labels=torch.zeros(winner.shape,device=winner.device,dtype=torch.int32); finite=torch.isfinite(best); labels[finite]=winner[finite].to(torch.int32)+1; return labels.detach().cpu().numpy()
def fragmentation_metrics(predicted,tag):
    crop=source9_crop_slices(); pred=predicted[crop]; gt=gt_labels_native[crop]; rows=[]
    for gid in source9_gt_ids.tolist():
        gm=gt==gid; gc=int(gm.sum()); overlapping=np.unique(pred[gm]); overlapping=overlapping[overlapping>0]; sig=[]; best=0.
        for pid in overlapping.tolist():
            pm=pred==pid; inter=int(np.count_nonzero(gm&pm));
            if inter>=max(3,int(.05*max(gc,1))): sig.append(pid)
            best=max(best,2*inter/max(int(gm.sum())+int(pm.sum()),1))
        rows.append({'tag':tag,'gt_id':gid,'significant_predicted_fragments':len(sig),'best_composed_dice':best,'missing':int(len(sig)==0)})
    f=pd.DataFrame(rows); s={'tag':tag,'mean_best_composed_dice':float(f.best_composed_dice.mean()),'missing_gt':int(f.missing.sum()),'gt_with_multiple_fragments':int((f.significant_predicted_fragments>1).sum()),'mean_fragments_per_gt':float(f.significant_predicted_fragments.mean()),'predicted_instance_count_in_crop':int(np.count_nonzero(np.unique(pred)>0))}; return f,s
def _composition():
    oq=source9_pairing['query_rows'].to(device); pq=valid_proposal_query_rows(out_spatial); refs=out_spatial.query_initial_references_cellscale[0,pq].detach().float().cpu(); scores=out_spatial.exist_logits[0,pq].sigmoid().detach().float().cpu(); d=torch.cdist(refs,source9_gt_centers.float()); keep=(d.min(dim=1).values<=SOURCE9_SELECTION_NEIGHBORHOOD_DREF)&(scores>=DEFAULT_EXIST_THRESHOLD); lq=pq[keep.to(pq.device)]
    if len(lq)>MAX_SOURCE9_RENDER_QUERIES: lq=lq[torch.topk(out_spatial.exist_logits[0,lq].sigmoid(),MAX_SOURCE9_RENDER_QUERIES).indices]
    ol=compose_queries(out_spatial,oq); ll=compose_queries(out_spatial,lq); of,os=fragmentation_metrics(ol,'oracle_one_per_gt')
    if ll is not None: lf,ls=fragmentation_metrics(ll,'learned_existence_selection')
    else: lf=pd.DataFrame(); ls={'tag':'learned_existence_selection','mean_best_composed_dice':0.,'missing_gt':len(source9_gt_ids),'gt_with_multiple_fragments':0,'mean_fragments_per_gt':0.,'predicted_instance_count_in_crop':0}
    of.to_csv(RUN_DIR/'AUX_oracle_composition_per_cell.csv',index=False); lf.to_csv(RUN_DIR/'AUX_learned_composition_per_cell.csv',index=False); crop=source9_crop_slices(); np.savez_compressed(RUN_DIR/'source9_composition.npz',gt_source9_crop=gt_labels_native[crop].astype(np.int32),current_source9_crop=current_labels_native[crop].astype(np.int32),oracle_composed_crop=ol[crop].astype(np.int32),learned_composed_crop=ll[crop].astype(np.int32) if ll is not None else np.zeros_like(gt_labels_native[crop],dtype=np.int32),oracle_query_rows=oq.detach().cpu().numpy(),learned_query_rows=lq.detach().cpu().numpy()); return of,os,lf,ls
composition_result=safe_phase('AUX_source9_composition',_composition)
if composition_result is not None:
    composition_oracle_df,composition_oracle_summary,composition_learned_df,composition_learned_summary=composition_result; display(composition_oracle_df)
    if len(composition_learned_df): display(composition_learned_df)
    print('Oracle:',composition_oracle_summary); print('Learned:',composition_learned_summary); record_result('AUX_oracle_composition',**composition_oracle_summary); record_result('AUX_learned_composition',**composition_learned_summary)
cleanup()


# Gate 8 — Finally add temporal information

This is intentionally a **screen, not a final temporal verdict**. The short training above was spatial-only. If the warm-start checkpoint predates the newest event-routing parameters, those parameters may still be close to initialization.

We compare the same current model with empty temporal state vs the real temporal graph/event data and measure source-9 proposal recall, paired center geometry, learned selection, and local-mask quality under GT-aware one-per-cell pairing.


In [ ]:
def make_full_device_batch_reusing_spatial():
    result={}; temporal_keys=NODE_KEYS|EDGE_INDEX_KEYS|EDGE_ATTR_KEYS|TRACKLET_KEYS|HYP_EDGE_INDEX_KEYS|HYP_EDGE_ATTR_KEYS
    for key,value in batch_cpu.items():
        if key=='targets': result[key]=value
        elif key not in temporal_keys and key in b_spatial: result[key]=b_spatial[key]
        else: result[key]=move_to_device(value,device)
    result['spatial_inputs']=result['spatial_inputs'].to(dtype=AMP_DTYPE); result['instance_labels']=result['instance_labels'].to(dtype=torch.int32); return result

def quick_output_summary(outputs,tag):
    pairing=build_gt_aware_pairing(outputs,source9_gt_ids,source9_gt_centers); q=pairing['query_rows']; g=pairing['gt_rows']; qd=q.to(device); s={'tag':tag}
    if len(q):
        gt=source9_gt_centers[g]; initial=outputs.query_initial_references_cellscale[0,qd].detach().float().cpu(); final=outputs.centers_cellscale[0,qd].detach().float().cpu(); s['paired_initial_center_mean_dref']=float(torch.linalg.vector_norm(initial-gt,dim=-1).mean()); s['paired_final_center_mean_dref']=float(torch.linalg.vector_norm(final-gt,dim=-1).mean()); _,ms=eval_local_masks(trainer.model.local_mask_decoder,outputs,pairing['gt_ids'],outputs.query_initial_references_cellscale[0,qd].detach(),outputs.query_embeddings[0,qd].detach(),f'{tag}_local_masks'); s['paired_local_hard_dice_mean']=ms['hard_dice_mean']; s['paired_local_hard_dice_min']=ms['hard_dice_min']
    if outputs.proposals is not None:
        valid=~outputs.proposals.padding_mask[0]; refs=outputs.proposals.references_cellscale[0,valid].detach().float().cpu(); s.update(proposal_set_metrics(refs,source9_gt_centers,'source9'))
    sel=selection_metrics_for_source9(outputs,DEFAULT_EXIST_THRESHOLD); s.update({f'selection_{k}':v for k,v in sel.items() if k!='threshold'}); return s

temporal_summary_df=pd.DataFrame(); out_full=None
def _gate8():
    bfull=make_full_device_batch_reusing_spatial(); trainer.model.eval(); cleanup(); torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(),torch.autocast(device_type='cuda',dtype=AMP_DTYPE): out=model_forward_from_batch(trainer.model,bfull,return_debug=True,temporal_memory_ablation='full',temporal_routing_ablation='full')
    frame=pd.DataFrame([quick_output_summary(out_spatial,'spatial_only'),quick_output_summary(out,'full_temporal')]); frame.to_csv(RUN_DIR/'G8_temporal_screen.csv',index=False); return out,frame
gate8_result=safe_phase('G8_temporal_screen',_gate8)
if gate8_result is not None: out_full,temporal_summary_df=gate8_result; display(temporal_summary_df); record_result('G8_temporal_screen',rows=temporal_summary_df.to_dict('records'))
cleanup()


# Automatic causal diagnosis

In [ ]:
diagnosis_rows=[]
def add_diag(stage,verdict,evidence,next_action): diagnosis_rows.append({'stage':stage,'verdict':verdict,'evidence':evidence,'next_action':next_action})
if gate1_summary is None: add_diag('G1 local mask capacity','NOT INTERPRETABLE','Oracle local-mask fit failed.','Fix Gate 1 before changing upstream architecture.')
else:
    hm=gate1_summary['hard_dice_mean']; hmin=gate1_summary['hard_dice_min']; add_diag('G1 local mask capacity','PASS' if hm>=.85 and hmin>=.60 else 'FAIL',f'GT anchors + common query: mean hard Dice {hm:.3f}, min {hmin:.3f}.','Move to learned query behavior.' if hm>=.85 and hmin>=.60 else 'Stop upstream debugging; reconsider local mask evidence/decoder/support.')
if gate2_summary is None: add_diag('G2 learned query at GT anchor','NOT INTERPRETABLE','Gate 2 failed.','Fix Gate 2.')
else:
    h2=gate2_summary['hard_dice_mean']; drop=gate1_summary['hard_dice_mean']-h2 if gate1_summary else float('nan'); good=h2>=.70 and (not math.isfinite(drop) or drop<=.15); add_diag('G2 learned query at GT anchor','PASS' if good else 'SUPPORTED PROBLEM',f'Hard Dice {h2:.3f}; oracle gap {drop:.3f}.','Test proposal anchors.' if good else 'The trained query/local-decoder interface is weak even with perfect centers.')
if gate3_summary is None: add_diag('G3 predicted proposal anchors','NOT INTERPRETABLE','Gate 3 failed.','Fix Gate 3.')
else:
    h3=gate3_summary['hard_dice_mean']; cov=gate3_summary['support_coverage_min']; h2=gate2_summary['hard_dice_mean'] if gate2_summary else float('nan'); drop=h2-h3 if math.isfinite(h2) else float('nan'); good=h3>=.65 and cov>=.98 and (not math.isfinite(drop) or drop<=.10); add_diag('G3 predicted proposal anchors','PASS' if good else 'SUPPORTED PROBLEM',f'Hard Dice {h3:.3f}, min support coverage {cov:.3f}, GT-anchor drop {drop:.3f}.','Proposal location is sufficient; inspect center/cardinality.' if good else 'Proposal localization/support is materially degrading masks.')
    ie=gate3_summary['initial_center_error_mean_dref']; fe=gate3_summary['final_center_error_mean_dref']; goodc=fe<=ie+.05; add_diag('G3b center refinement identity','PASS' if goodc else 'SUPPORTED PROBLEM',f'Paired center error {ie:.3f} -> {fe:.3f} dref.','Anchor-relative center refinement is acceptable.' if goodc else 'Center refinement still moves correct proposals away from their own GT.')
if gate4_summary is None: add_diag('G4 proposal coverage','NOT INTERPRETABLE','Proposal gate failed.','Fix proposal diagnostics.')
else:
    s9=gate4_summary.get('source9_all_proposals_recall_0p5',float('nan')); ag=gate4_summary.get('all_gt_all_proposals_recall_0p5',float('nan')); good=s9>=.999 and ag>=.90; add_diag('G4 proposal coverage','PASS' if good else 'SUPPORTED PROBLEM',f'Recall@0.5 source9={s9:.3f}, all GT={ag:.3f}.','Focus on selection/cardinality.' if good else 'Proposal pool itself is missing usable cell hypotheses.')
if gate5_summary is None: add_diag('G5 learned cardinality','NOT INTERPRETABLE','Selection gate failed.','Fix selection diagnostic.')
else:
    unique=int(gate5_summary['unique_gt_covered']); missing=int(gate5_summary['missing_gt']); dup=int(gate5_summary['duplicate_selected']); good=unique==len(source9_gt_ids) and missing==0 and dup<=1; add_diag('G5 learned cardinality','PASS' if good else 'SUPPORTED PROBLEM',f'Default threshold covers {unique}/{len(source9_gt_ids)}, missing={missing}, duplicate survivors={dup}.','Existence selection is provisionally adequate.' if good else 'Cardinality/existence selection is a direct candidate for multi-color fragmentation.')
if gate6_summary is None: add_diag('G6 proposal score field / NMS','NOT INTERPRETABLE','Score-field gate failed.','Fix score-field diagnostic.')
else:
    o=gate6_summary.get('oracle_score_nms_all_gt_recall_0p5',float('nan')); l=gate6_summary.get('learned_score_peaks_all_gt_recall_0p5',float('nan'))
    if o<.99: add_diag('G6 proposal score field / NMS','SUPPORTED PROBLEM',f'Oracle score NMS recall@0.5={o:.3f}.','NMS/extraction geometry suppresses legitimate cells.')
    elif l+.05<o: add_diag('G6 proposal score field / NMS','SUPPORTED PROBLEM',f'Oracle NMS recall={o:.3f}, learned peak recall={l:.3f}.','NMS can work; learned proposal score field is the issue.')
    else: add_diag('G6 proposal score field / NMS','PASS',f'Oracle NMS recall={o:.3f}, learned peak recall={l:.3f}.','Proposal score/extraction is provisionally adequate.')
if gate7_summary is not None:
    if gate7_summary['missing_gt_count']>0:
        f=gate7_summary['missing_gt_with_selected_proposal_within_1dref']; t=gate7_summary['missing_gt_count']; add_diag('G7a missing-cell discovery','PASS' if f==t else 'SUPPORTED PROBLEM',f'Selected nearby proposal for {f}/{t} GT cells absent from current mask.','Off-mask discovery works here.' if f==t else 'Model still misses some cells with no current-mask support.')
    if gate7_summary['noisy_source_count']>0:
        s=gate7_summary['noisy_sources_with_selected_query']; t=gate7_summary['noisy_source_count']; add_diag('G7b noisy-current suppression','PASS' if s==0 else 'SUPPORTED PROBLEM',f'{s}/{t} GT-empty current components still have selected proposal queries.','Noisy suppression works here.' if s==0 else 'Existence reasoning still trusts noisy current components.')
if composition_oracle_summary is not None and composition_learned_summary is not None:
    of=composition_oracle_summary['gt_with_multiple_fragments']; lf=composition_learned_summary['gt_with_multiple_fragments']; add_diag('Auxiliary final composition','SUPPORTED PROBLEM' if lf>of else 'PASS',f'Multiple-fragment GT cells: oracle selection={of}, learned selection={lf}.','Learned overcomplete survival contributes to colored fragmentation.' if lf>of else 'Selection does not add much fragmentation beyond mask quality.')
if len(temporal_summary_df)==2:
    sr=temporal_summary_df.iloc[0]; tr=temporal_summary_df.iloc[1]; sm=float(sr.get('paired_local_hard_dice_mean',float('nan'))); tm=float(tr.get('paired_local_hard_dice_mean',float('nan'))); sc=float(sr.get('paired_final_center_mean_dref',float('nan'))); tc=float(tr.get('paired_final_center_mean_dref',float('nan'))); add_diag('G8 temporal information','SCREEN ONLY',f'Local hard Dice spatial={sm:.3f}, temporal={tm:.3f}; final center error spatial={sc:.3f}, temporal={tc:.3f} dref.','Do not make a temporal architecture verdict from short spatial-only training; use the overnight-trained checkpoint later.')
diagnosis_df=pd.DataFrame(diagnosis_rows); display(diagnosis_df); diagnosis_df.to_csv(RUN_DIR/'diagnosis.csv',index=False); (RUN_DIR/'diagnosis.json').write_text(json.dumps(diagnosis_rows,indent=2),encoding='utf-8')
priority=diagnosis_df[diagnosis_df.verdict.isin(['FAIL','SUPPORTED PROBLEM'])]
if len(priority):
    p=priority.iloc[0]; print('\n'+'='*90+'\nFIRST PRIORITY FAILURE / SUPPORTED PROBLEM\n'+'='*90); print(p.stage); print(p.evidence); print('Next:',p.next_action)
else: print('No strong spatial failure isolated by completed gates; inspect detailed tables.')


## Final run-health report

In [ ]:
elapsed=(time.monotonic()-NOTEBOOK_STARTED)/60; health={'git_head':HEAD,'elapsed_minutes':elapsed,'minutes_left_from_hard_budget':minutes_left(),'training_rows':len(training_rows),'errors_logged':sum(1 for _ in ERRORS_JSONL.open('r',encoding='utf-8')) if ERRORS_JSONL.exists() else 0,'run_dir':str(RUN_DIR),'warm_start':warm_start_info}
(RUN_DIR/'run_health.json').write_text(json.dumps({k:_jsonable(v) for k,v in health.items()},indent=2),encoding='utf-8'); (RUN_DIR/'RUN_COMPLETE.json').write_text(json.dumps({'completed_at':time.strftime('%Y-%m-%d %H:%M:%S'),'elapsed_minutes':elapsed},indent=2),encoding='utf-8')
print('='*90); print('NOTEBOOK 29 COMPLETE'); print('='*90); print(json.dumps({k:_jsonable(v) for k,v in health.items()},indent=2)); print('\nKey outputs:')
for name in ['training.csv','G1_oracle_mask_per_cell.csv','G2_actual_query_gt_anchor.csv','G3_predicted_anchor_masks.csv','G3_center_identity.csv','G4_proposal_location_summary.csv','G5_selection_threshold_sweep.csv','G6_score_field_summary.csv','G7_missing_cell_discovery.csv','G7_noisy_component_suppression.csv','source9_composition.npz','G8_temporal_screen.csv','diagnosis.csv','errors.jsonl']:
    p=RUN_DIR/name
    if p.exists(): print(' -',p)


## Optional Napari inspection
The notebook does not open a GUI while unattended. `source9_composition.npz` contains compact GT/current/oracle-selection/learned-selection crops for later inspection.

In [ ]:
if OPEN_NAPARI_AT_END:
    try:
        import napari
        payload=np.load(RUN_DIR/'source9_composition.npz'); viewer=napari.Viewer(title='STIR-Net Notebook 29 — source9 oracle ladder')
        viewer.add_labels(payload['gt_source9_crop'],name='GT',scale=tuple(spacing_native)); viewer.add_labels(payload['current_source9_crop'],name='Current segmentation',scale=tuple(spacing_native)); viewer.add_labels(payload['oracle_composed_crop'],name='Oracle one-per-GT composition',scale=tuple(spacing_native)); viewer.add_labels(payload['learned_composed_crop'],name='Learned existence composition',scale=tuple(spacing_native)); viewer.dims.ndisplay=3
    except BaseException as exc: record_error('optional_napari',exc)
